# TerraPulse AI · 02: LoRA fine-tune Qwen3-1.7B (Colab T4)
Open in VS Code via the **Colab extension**, runtime **T4 GPU**. Needs `training/data/intent.jsonl` and
`training/data/compose.jsonl` from notebook 01 (re-run 01 in this runtime, or download them from the HF dataset).

Base: `Qwen/Qwen3-1.7B` (Apache-2.0, ~2.0B params incl. embeddings, under the 3B limit). LoRA r=16, α=32, 2 epochs, completion-only loss.
Outputs pushed to the Hub: adapter, merged safetensors (`from_pretrained` works), GGUF Q4_K_M + Q8_0 (what the API serves).

In [ ]:
# Live log: copy everything this notebook prints (training steps, scores, uploads) to ~/round2_live.log,
# so a headless run (run_round2.sh) can be followed with:  tail -f ~/round2_live.log
import os, sys
class _Tee:
    def __init__(self, stream, fh): self.stream, self.fh = stream, fh
    def write(self, s):
        self.stream.write(s); self.fh.write(s); self.fh.flush(); return len(s)
    def flush(self): self.stream.flush(); self.fh.flush()
    def __getattr__(self, name): return getattr(self.stream, name)
_live = open(os.path.expanduser("~/round2_live.log"), "a", buffering=1)
sys.stdout, sys.stderr = _Tee(sys.stdout, _live), _Tee(sys.stderr, _live)
print("=== notebook 02 started ===")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the project into this runtime: Colab or any Jupyter server (e.g. a college DGX).
# Option A: clone your GitHub repo. Option B (Colab only): upload GEOAI.zip.
REPO_URL = "https://github.com/aaron43210/GeoAI_Hackathon.git"   # set "" to upload GEOAI.zip instead
REPO_BRANCH = "round2-dgx"       # round-2 work lives on this branch; main stays as the submitted version
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
WORK = "/content/GEOAI" if IN_COLAB else os.path.expanduser("~/GeoAI_Hackathon")   # its own folder (change if home is small)
if REPO_URL and os.path.isdir(WORK) and not os.path.exists(f"{WORK}/.git"):
    raise SystemExit(f"{WORK} exists but is not a git clone of the repo: rename/remove it or change WORK")
# GPUs: on a shared multi-GPU server, use the ones with the most free memory (set CUDA_VISIBLE_DEVICES yourself to override).
NUM_GPUS = 1                           # Unsloth (open-source) trains on one GPU; a 1.7B LoRA needs only one
def _free_gpus(min_free_mb=20000):
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=True).stdout
    except Exception:
        return []
    gpus = [tuple(int(x) for x in l.split(",")) for l in out.strip().splitlines() if l.strip()]
    return [i for i, free in sorted(gpus, key=lambda g: -g[1]) if free >= min_free_mb]
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")     # same GPU numbering as nvidia-smi
if "CUDA_VISIBLE_DEVICES" not in os.environ and not IN_COLAB:
    os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in _free_gpus()[:NUM_GPUS]) or "0"
print("GPU(s):", os.environ.get("CUDA_VISIBLE_DEVICES", "default"))
if REPO_URL and os.path.exists(f"{WORK}/.git"):      # existing clone: switch it to REPO_BRANCH at its latest commit
    subprocess.run(["git", "-C", WORK, "fetch", "--depth", "1", "origin",
                    f"+{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}"], check=True)
    subprocess.run(["git", "-C", WORK, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
elif REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, WORK], check=True)
elif not os.path.exists(WORK):
    from google.colab import files
    up = files.upload()                         # choose GEOAI.zip (project folder zipped, without node_modules)
    subprocess.run(["unzip", "-q", next(iter(up)), "-d", os.path.dirname(WORK)], check=True)
os.chdir(WORK)
print("working in", os.getcwd())
%pip -q install -r backend/requirements-core.txt
sys.path.insert(0, f"{WORK}/backend")

In [ ]:
# Smart install: if unsloth + cu121 torch are already installed (pre-installed in terminal),
# skip ALL pip calls. This prevents unsloth from reinstalling torch cu130 inside the kernel.
import subprocess, sys, importlib.util

def _already_good():
    if importlib.util.find_spec('unsloth') is None: return False
    r = subprocess.run([sys.executable,'-m','pip','show','torch'],capture_output=True,text=True)
    return any(v in r.stdout for v in ['2.5.1+cu121','2.5.1+cu122','2.5.1+cu124'])

if _already_good():
    print('Pre-installed: unsloth + torch cu121 found. Skipping pip install.')
else:
    # Write a pip constraint to prevent unsloth from upgrading torch
    open('/tmp/torch_pin.txt','w').write('torch==2.5.1\n')
    r = subprocess.run([sys.executable,'-m','pip','show','torch'],capture_output=True,text=True)
    if not any(v in r.stdout for v in ['cu121','cu122','cu124']):
        print('Installing PyTorch cu121 for CUDA 12.2 driver...')
        subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall',
            'torch==2.5.1','torchvision','torchaudio',
            '--index-url','https://download.pytorch.org/whl/cu121'],check=True)
        open('/tmp/torch_pin.txt','w').write('torch==2.5.1\n')
    # Install unsloth WITH constraint so it cannot upgrade torch
    subprocess.run([sys.executable,'-m','pip','install','-q',
        'unsloth','trl>=0.20','datasets','-c','/tmp/torch_pin.txt'],check=True)

import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# Hugging Face login: uses the Colab secret HF_TOKEN if available, otherwise asks (input is hidden, never saved).
import os
from huggingface_hub import login
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    from getpass import getpass
    token = getpass("Hugging Face write token: ")
login(token)
os.environ["HF_TOKEN"] = token
HF_USER = "aaronr43210"

In [ ]:
import os, json, random
if not os.path.exists("training/data/compose.jsonl"):
    from huggingface_hub import hf_hub_download
    os.makedirs("training/data", exist_ok=True)
    for f in ["intent.jsonl", "compose.jsonl"]:
        p = hf_hub_download(f"{HF_USER}/terrapulse-sft-data", f, repo_type="dataset"); os.system(f"cp {p} training/data/{f}")
from app.slm.runtime import render_chatml
rows = [json.loads(l) for f in ["intent.jsonl", "compose.jsonl"] for l in open(f"training/data/{f}")]
random.Random(0).shuffle(rows)
def to_pc(r):   # prompt = exactly what the runtime sends (ChatML + empty <think>), completion = assistant text
    return {"prompt": render_chatml(r["messages"][:-1]), "completion": r["messages"][-1]["content"] + "<|im_end|>",
            "task": r["task"]}
data = [to_pc(r) for r in rows]
# Hold out the last 20 live fact sheets for the grounding eval and keep them OUT of training (no leakage).
_held = {json.loads(l)["fact_sheet"] for l in list(open("training/data/factsheets.jsonl"))[-20:]} \
    if os.path.exists("training/data/factsheets.jsonl") else set()
data = [d for d in data if not any(h in d["prompt"] for h in _held)]
print("held out", len(_held), "fact sheets for evaluation")
n_val = max(40, len(data) // 20)
val, train = data[:n_val], data[n_val:]
print(len(train), "train /", len(val), "val;", sum(d["task"] == "compose" for d in train), "compose")

In [ ]:
from unsloth import FastLanguageModel
import torch
BASE = "Qwen/Qwen3-1.7B"
model, tokenizer = FastLanguageModel.from_pretrained(BASE, max_seq_length=2048, load_in_4bit=False, dtype=None)

### Baseline eval (before fine-tuning): intent gold set + grounding pass rate

In [ ]:
from app.schemas.intent import Intent
from app.slm.prompts import intent_messages
from app.evidence.grounding import verify
import re, time
gold = [json.loads(l) for l in open("backend/eval/intent_gold.jsonl")]
sheets = [json.loads(l) for l in open("training/data/factsheets.jsonl")][-20:] if os.path.exists("training/data/factsheets.jsonl") else []

def gen(prompt, max_new=300):
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluate(tag):
    FastLanguageModel.for_inference(model)
    acc = {"activity": 0, "location": 0, "profiles": 0, "in_scope": 0, "json_ok": 0}
    for g in gold:
        txt = gen(render_chatml(intent_messages(g["q"])), 260)
        try:
            m = re.search(r"\{.*\}", txt, re.S); p = Intent(**json.loads(m.group(0))).normalised(); acc["json_ok"] += 1
        except Exception:
            continue
        acc["activity"] += p.activity == g["activity"] or (not g["in_scope"] and not p.in_scope)
        loc = (p.location_text or "").lower()
        acc["location"] += (g["location"] is None and not loc) or (g["location"] is not None and bool(loc) and (g["location"] in loc or loc in g["location"]))
        acc["profiles"] += sorted(p.profiles) == sorted(g["profiles"])
        acc["in_scope"] += p.in_scope == g["in_scope"]
    res = {k: round(v / len(gold), 3) for k, v in acc.items()}
    passed = 0
    from app.slm.prompts import chat_messages
    for s in sheets:
        a = gen(render_chatml(chat_messages(s["fact_sheet"])), 420)
        passed += verify(a, s["fact_sheet"], set(s["evidence_ids"]), s["known_names"], s["must_mention"], s["recommended"], s.get("evidence_values")).status == "passed"
    res["grounding_pass"] = round(passed / max(1, len(sheets)), 3)
    print(tag, res); return res
before = evaluate("BEFORE")

In [ ]:
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(model=model, processing_class=tokenizer,
    train_dataset=Dataset.from_list([{k: d[k] for k in ("prompt", "completion")} for d in train]),
    eval_dataset=Dataset.from_list([{k: d[k] for k in ("prompt", "completion")} for d in val]),
    args=SFTConfig(output_dir="outputs", num_train_epochs=2, per_device_train_batch_size=4, gradient_accumulation_steps=4,
                   learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=10,
                   eval_strategy="steps", eval_steps=100, save_strategy="no", max_length=2048,
                   completion_only_loss=True, fp16=not torch.cuda.is_bf16_supported(),
                   bf16=torch.cuda.is_bf16_supported(), report_to="none", seed=0))
stats = trainer.train(); print(stats)

In [ ]:
after = evaluate("AFTER")
print(json.dumps({"before": before, "after": after}, indent=1))

### Publish: adapter, merged model (`from_pretrained`), GGUF for CPU serving, model card

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
# Round 2 goes to separate "-v2" repos, so the model the live Space uses (round 1) is never overwritten.
# Switch the Space to -v2 only after testing it; set SUFFIX = "" to publish over round 1 instead.
SUFFIX = "-v2"
LORA, MERGED, GGUF = (f"{HF_USER}/terrapulse-qwen3-1.7b-lora{SUFFIX}", f"{HF_USER}/terrapulse-qwen3-1.7b{SUFFIX}",
                      f"{HF_USER}/terrapulse-qwen3-1.7b-GGUF{SUFFIX}")
LOCAL = "training/runs/lora_local"                     # local copy first, so a failed upload loses nothing
model.save_pretrained(LOCAL); tokenizer.save_pretrained(LOCAL); print("adapter saved to", LOCAL)
for r in (LORA, MERGED, GGUF): api.create_repo(r, exist_ok=True)
model.push_to_hub(LORA); tokenizer.push_to_hub(LORA)
model.push_to_hub_merged(MERGED, tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method=["q4_k_m", "q8_0"])
import glob
for f in glob.glob("gguf_out*/*.gguf") + glob.glob("gguf_out/*.gguf"):
    q = "Q4_K_M" if "q4_k_m" in f.lower() else ("Q8_0" if "q8_0" in f.lower() else None)
    if q: api.upload_file(path_or_fileobj=f, path_in_repo=f"terrapulse-qwen3-1.7b.{q}.gguf", repo_id=GGUF)
print("uploaded")

In [ ]:
card = open("training/MODEL_CARD.md").read()
table = "| metric | base Qwen3-1.7B | TerraPulse LoRA |\n|---|---|---|\n" + "\n".join(
    f"| {k} | {before[k]} | {after[k]} |" for k in before)
_teachers = sorted({json.loads(l).get("teacher") or "Qwen/Qwen3-4B" for l in open("training/data/compose.jsonl")
                    if json.loads(l).get("source") == "teacher"})   # rows without the field come from round 1 (Qwen3-4B)
TEACHERS = ", ".join(f"`{t}`" for t in _teachers)
HARDWARE = "a free Colab T4" if "google.colab" in sys.modules else f"one {torch.cuda.get_device_name(0)} GPU"
card = (card.replace("{{EVAL_TABLE}}", table).replace("{{N_TRAIN}}", str(len(train)))
            .replace("{{TEACHER}}", TEACHERS).replace("{{HARDWARE}}", HARDWARE))
print("model card: teacher", TEACHERS, "| hardware", HARDWARE)
for r in (LORA, MERGED, GGUF):
    api.upload_file(path_or_fileobj=card.replace("{{REPO}}", r).encode(), path_in_repo="README.md", repo_id=r)
open("training/eval_results.json", "w").write(json.dumps({"before": before, "after": after}, indent=1))
print(table)

In [ ]:
# Judges' check: the merged model loads with from_pretrained
from transformers import AutoModelForCausalLM, AutoTokenizer
m = AutoModelForCausalLM.from_pretrained(MERGED, torch_dtype="auto", device_map="auto"); t = AutoTokenizer.from_pretrained(MERGED)
print(sum(p.numel() for p in m.parameters()) / 1e9, "B params, loaded OK")